# 7-bob. 7-BOB. ZARARLI DASTURLARNI ANIQLASH I: STATIK TAHLIL
## Amaliy misol: statik xususiyatlar asosidagi detektor

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/07-bob/7-3_statik-xususiyatlar-asosidagi-detektor.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 7-bob.*

Quyidagi dasturda 10 ming ta faylning beshta statik xususiyati sun'iy ravishda yaratiladi: entropiya, import qilingan funksiyalar soni, xavfli funksiyalar soni, bo'limlar soni va raqamli imzo mavjudligi. Xususiyatlarning qiymatlari 7.4-jadvaldagi qonuniyatlarga mos ravishda tanlangan, biroq zararli va zararsiz fayllar to'liq ajralmaydi: real hayotdagi kabi, ba'zi zararsiz fayllar ham paketlangan, ba'zi zararli fayllar esa imzolangan bo'lishi mumkin. Model gradiyentli kuchaytirish usuli bilan o'qitiladi va past noto'g'ri signallar ulushida baholanadi. Dastur amaliy mashg'ulotning A7 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score

rng = np.random.default_rng(5)

def fayllar(soni, zararli):
    """Har bir fayl uchun 5 ta statik xususiyat (sun'iy ma'lumot)."""
    if zararli:
        entropiya   = np.clip(rng.normal(6.6, 0.9, soni), 0, 8)   # ko'pincha paketlangan
        importlar   = rng.poisson(70, soni)                      # import qilingan funksiyalar
        xavfli_api  = rng.poisson(2.5, soni)                     # xavfli funksiyalar soni
        bolimlar    = rng.integers(3, 10, soni)                  # bo'limlar soni
        imzo        = rng.random(soni) < 0.15                    # raqamli imzo bormi
    else:
        entropiya   = np.clip(rng.normal(6.0, 0.9, soni), 0, 8)
        importlar   = rng.poisson(90, soni)
        xavfli_api  = rng.poisson(1, soni)
        bolimlar    = rng.integers(3, 7, soni)
        imzo        = rng.random(soni) < 0.50
    return np.column_stack([entropiya, importlar, xavfli_api, bolimlar, imzo])

X = np.vstack([fayllar(5000, False), fayllar(5000, True)])
y = np.array([0] * 5000 + [1] * 5000)
X_o, X_s, y_o, y_s = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

model = HistGradientBoostingClassifier(random_state=0).fit(X_o, y_o)
ball = model.predict_proba(X_s)[:, 1]

# Antivirus uchun asosiy savol: noto'g'ri signallar juda kam bo'lganda
# zararli fayllarning qancha qismini aniqlay olamiz?
fpr, tpr, _ = roc_curve(y_s, ball)
for chegara_fpr in [0.01, 0.001]:
    print(f"Noto'g'ri signallar ulushi {chegara_fpr:.1%} bo'lganda to'liqlik: "
          f"{tpr[fpr <= chegara_fpr].max():.1%}")

print(f"ROC egri chizig'i ostidagi yuza (AUC): {roc_auc_score(y_s, ball):.3f}")

Noto'g'ri signallar ulushi 1.0% bo'lganda to'liqlik: 76.6%
Noto'g'ri signallar ulushi 0.1% bo'lganda to'liqlik: 60.0%
ROC egri chizig'i ostidagi yuza (AUC): 0.980


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Noto'g'ri signallar ulushi 1.0% bo'lganda to'liqlik: 76.6%
Noto'g'ri signallar ulushi 0.1% bo'lganda to'liqlik: 60.0%
ROC egri chizig'i ostidagi yuza (AUC): 0.980
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 7-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.